<a href="https://colab.research.google.com/github/annasvenbro/etudesnordiques/blob/main/20250926_MAP_BnF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests as rq
from bs4 import BeautifulSoup as bs
import pandas as pd
from pandas import plotting
import matplotlib.pyplot as plt

In [ ]:
def get_bnf(annee):
  url=rq.get(f"https://catalogue.bnf.fr/api/SRU?version=1.2&operation=searchRetrieve&query=(bib.language%20all%20%22nor%22)%20or%20(bib.language%20all%20%22nob%22)%20or%20(bib.language%20all%20%22nno%22)%20and%20(bib.language%20all%20%22fre%22)%20and%20(bib.publicationdate%20all%20%22{annee}%22)&recordSchema=unimarcXchange&maximumRecords=1000&startRecord=1")#On insère la variable annee dans une f-string avec l'API.
  soupe=bs(url.text, "xml")
  records=soupe.find_all("srw:record")
  compteur=0
  for record in records:
    try:
        datafields = record.find_all("mxc:datafield", attrs={"tag": "101", "ind1": "1", "ind2": " "})
        for datafield in datafields:
            subfields=datafield.find_all("mxc:subfield", attrs={"code": "a"})
            for subfield in subfields:
                if subfield.text=="fre":
                    subfield_c = datafield.find("mxc:subfield", attrs={"code": "c"})
                    if subfield_c and (subfield_c.text=="nor" or subfield_c.text=="nob" or subfield_c.text=="nno"):
                        compteur += 1
                        break
    except Exception:
        pass
  return compteur

In [ ]:
dict_bnf={}
for annee in range(1900,1951) :
  publications=get_bnf(annee)
  dict_bnf[annee]=publications

df_bnf = pd.DataFrame.from_dict([dict_bnf])
df_bnf=df_bnf.transpose()
df_bnf=df_bnf.rename_axis("Date").reset_index()
df_bnf=df_bnf.set_index("Date")
df_bnf.columns=["Translations"]


ax = df_bnf.plot(
    figsize=(12,5),
    kind="bar",
    title="Translations from Norwegian to French (1900-1950) in BnF's General Catalogue"
)

fig = ax.figure
fig.savefig("traductions_bnf.png", dpi=300, bbox_inches="tight")

In [ ]:
def get_ark(annee):
  url=rq.get(f"https://catalogue.bnf.fr/api/SRU?version=1.2&operation=searchRetrieve&query=(bib.language%20all%20%22nor%22)%20or%20(bib.language%20all%20%22nob%22)%20or%20(bib.language%20all%20%22nno%22)%20and%20(bib.language%20all%20%22fre%22)%20and%20(bib.publicationdate%20all%20%22{annee}%22)&recordSchema=unimarcXchange&maximumRecords=1000&startRecord=1")#On insère la variable annee dans une f-string avec l'API.
  soupe=bs(url.text,"xml")
  records=soupe.find_all("srw:record")
  liste_ark=[]
  for record in records:
    datafield_101=record.find("datafield",{"tag":"101"})
    if datafield_101:
      subfield_a=datafield_101.find("subfield",{"code":"a"})
      if subfield_a and subfield_a.text=="fre":
        c_subfields=record.find_all("subfield",{"code":"c"})
        if any(c_subfield.text in ["nor","nob","nno"] for c_subfield in c_subfields):
          controlfield_003=record.find("controlfield",{"tag":"003"})
          if controlfield_003 is not None:
            liste_ark.append(controlfield_003.text)
  return liste_ark

In [ ]:
dict_ark={}
for annee in range(1900,1951) :
  dict_ark[annee]=get_ark(annee)
dict_ark

df_ark=pd.DataFrame.from_dict([dict_ark])
df_ark=df_ark.transpose()
df_ark=df_ark.rename_axis("Date").reset_index()
df_ark=df_ark.set_index("Date")
df_ark.columns=["Notices du catalogue"]

df_ark_exploded = df_ark.explode("Notices du catalogue").reset_index()

df_ark_exploded = df_ark_exploded.dropna(subset=["Notices du catalogue"])

In [ ]:
def get_titre(ark_url: str) -> str:
    if pd.isna(ark_url):  # gestion du cas NaN
        return None

    # Extraire l'identifiant ark:/...
    ark_id = ark_url.split("catalogue.bnf.fr/")[-1]  # ex: "ark:/12148/cb43798111t"

    # Construire l’URL de requête SRU
    query_url = (
        "https://catalogue.bnf.fr/api/SRU"
        "?version=1.2&operation=searchRetrieve"
        f"&query=bib.persistentid%20all%20%22{ark_id}%22"
        "&recordSchema=unimarcxchange&maximumRecords=20&startRecord=1"
    )

    # Appel API
    resp = rq.get(query_url)
    if resp.status_code != 200:
        return None

    # Parsing XML
    soupe = bs(resp.text, "xml")

    # Récupérer le champ 200
    datafield_200 = soupe.find("mxc:datafield", {"tag": "200", "ind1": "1", "ind2": " "})
    if not datafield_200:
        return None

    # Sous-zone "a" (titre principal)
    titre_principal = datafield_200.find("mxc:subfield", {"code": "a"})
    titre = titre_principal.text.strip() if titre_principal else ""

    # Sous-zones "e" (compléments répétés)
    complements = [sf.text.strip() for sf in datafield_200.find_all("mxc:subfield", {"code": "e"})]

    # Construire le titre final
    if complements:
        titre_final = " : ".join([titre] + complements)
    else:
        titre_final = titre

    return titre_final

In [ ]:
df_ark_exploded["Titre"] = df_ark_exploded["Notices du catalogue"].apply(get_titre)

In [ ]:
def get_titre_original(ark_url: str) -> str:
    if pd.isna(ark_url):
        return None

    # Extraire l'identifiant ark:/...
    ark_id = ark_url.split("catalogue.bnf.fr/")[-1]

    # Construire l’URL de requête SRU
    query_url = (
        "https://catalogue.bnf.fr/api/SRU"
        "?version=1.2&operation=searchRetrieve"
        f"&query=bib.persistentid%20all%20%22{ark_id}%22"
        "&recordSchema=unimarcxchange&maximumRecords=20&startRecord=1"
    )

    resp = rq.get(query_url)
    if resp.status_code != 200:
        return None

    soupe = bs(resp.text, "xml")

    # Champ 454 sous-zone t (titre original)
    datafield_454 = soupe.find("mxc:datafield", {"tag": "454", "ind1": " ", "ind2": "1"})
    if not datafield_454:
        return None

    subfield_t = datafield_454.find("mxc:subfield", {"code": "t"})
    if subfield_t:
        return subfield_t.text.strip()

    return None

In [ ]:
df_ark_exploded["Titre original"] = df_ark_exploded["Notices du catalogue"].apply(get_titre_original)

In [ ]:
def get_editeur(ark_url: str) -> str:
    if pd.isna(ark_url):
        return None

    # Extraire l'identifiant ARK
    ark_id = ark_url.split("catalogue.bnf.fr/")[-1]

    # Construire l’URL de requête SRU
    query_url = (
        "https://catalogue.bnf.fr/api/SRU"
        "?version=1.2&operation=searchRetrieve"
        f"&query=bib.persistentid%20all%20%22{ark_id}%22"
        "&recordSchema=unimarcxchange&maximumRecords=20&startRecord=1"
    )

    resp = rq.get(query_url)
    if resp.status_code != 200:
        return None

    soupe = bs(resp.text, "xml")

    # 1) Chercher zone 210 $c (tous ind1/ind2)
    datafield_210 = soupe.find("mxc:datafield", {"tag": "210"})
    if datafield_210:
        subfields_c = [sf.text.strip() for sf in datafield_210.find_all("mxc:subfield", {"code": "c"})]
        if subfields_c:
            return "; ".join(subfields_c)

    # 2) Sinon, chercher zones 214 $c avec ind2=0,1,2
    datafields_214 = soupe.find_all("mxc:datafield", {"tag": "214"})
    for df in datafields_214:
        ind2 = df.get("ind2", "")
        if ind2 in ["0", "1", "2"]:
            subfields_c = [sf.text.strip() for sf in df.find_all("mxc:subfield", {"code": "c"})]
            if subfields_c:
                return "; ".join(subfields_c)

    return None

In [ ]:
df_ark_exploded["Éditeur"] = df_ark_exploded["Notices du catalogue"].apply(get_editeur)

In [ ]:
def get_respint(ark_url: str) -> str:
    if pd.isna(ark_url):
        return None

    # Extraire l'identifiant ARK
    ark_id = ark_url.split("catalogue.bnf.fr/")[-1]

    # Construire l’URL de requête SRU
    query_url = (
        "https://catalogue.bnf.fr/api/SRU"
        "?version=1.2&operation=searchRetrieve"
        f"&query=bib.persistentid%20all%20%22{ark_id}%22"
        "&recordSchema=unimarcxchange&maximumRecords=20&startRecord=1"
    )

    resp = rq.get(query_url)
    if resp.status_code != 200:
        return None

    soupe = bs(resp.text, "xml")

    # 1) Chercher zone 200 $f (tous ind1/ind2)
    datafield_200 = soupe.find("mxc:datafield", {"tag": "200"})
    if datafield_200:
        subfields_f = [sf.text.strip() for sf in datafield_200.find_all("mxc:subfield", {"code": "f"})]
        if subfields_f:
            return "; ".join(subfields_f)

    return None

In [ ]:
df_ark_exploded["Responsabilité principale"] = df_ark_exploded["Notices du catalogue"].apply(get_respint)

In [ ]:
def get_respint2(ark_url: str) -> str:
    if pd.isna(ark_url):
        return None

    # Extraire l'identifiant ARK
    ark_id = ark_url.split("catalogue.bnf.fr/")[-1]

    # Construire l’URL de requête SRU
    query_url = (
        "https://catalogue.bnf.fr/api/SRU"
        "?version=1.2&operation=searchRetrieve"
        f"&query=bib.persistentid%20all%20%22{ark_id}%22"
        "&recordSchema=unimarcxchange&maximumRecords=20&startRecord=1"
    )

    resp = rq.get(query_url)
    if resp.status_code != 200:
        return None

    soupe = bs(resp.text, "xml")

    # 1) Chercher zone 200 $g (tous ind1/ind2)
    datafield_200 = soupe.find("mxc:datafield", {"tag": "200"})
    if datafield_200:
        subfields_g = [sf.text.strip() for sf in datafield_200.find_all("mxc:subfield", {"code": "g"})]
        if subfields_g:
            return "; ".join(subfields_g)

    return None

In [ ]:
df_ark_exploded["Responsabilité secondaire"] = df_ark_exploded["Notices du catalogue"].apply(get_respint2)

In [ ]:
def get_trad_nom(ark_url: str):

    if pd.isna(ark_url):
        return None

    # Extraire l'identifiant ARK
    ark_id = ark_url.split("catalogue.bnf.fr/")[-1]

    # Construire l’URL de requête SRU
    query_url = (
        "https://catalogue.bnf.fr/api/SRU"
        "?version=1.2&operation=searchRetrieve"
        f"&query=bib.persistentid%20all%20%22{ark_id}%22"
        "&recordSchema=unimarcxchange&maximumRecords=20&startRecord=1"
    )

    resp = rq.get(query_url)
    if resp.status_code != 200:
        return None

    soupe = bs(resp.text, "xml")

    trad_fields = soupe.find_all("mxc:datafield", tag=lambda t: t in ["701", "702"])

    tradnom_list = []
    for field in trad_fields:
        code4 = field.find("mxc:subfield", {"code": "4"})
        if code4 and code4.text.strip() == "730":
            nom = field.find("mxc:subfield", {"code": "a"})
            prenom = field.find("mxc:subfield",{"code": "b"})
            dates = field .find("mxc:subfield", {"code": "f"})
            nom_text = nom.text.strip() if nom else ""
            prenom_text = prenom.text.strip() if prenom else ""
            dates_text = f" ({dates.text.strip()})" if dates else ""
            tradnom_text=f"{nom_text}, {prenom_text} {dates_text}"

            if nom and prenom:
                tradnom_list.append(tradnom_text)

    return tradnom_list

In [ ]:
df_ark_exploded["Nom Traducteurs"] = df_ark_exploded["Notices du catalogue"].apply(get_trad_nom)

In [ ]:
def get_trad_isni(ark_url: str):

    if pd.isna(ark_url):
        return None

    # Extraire l'identifiant ARK
    ark_id = ark_url.split("catalogue.bnf.fr/")[-1]

    # Construire l’URL de requête SRU
    query_url = (
        "https://catalogue.bnf.fr/api/SRU"
        "?version=1.2&operation=searchRetrieve"
        f"&query=bib.persistentid%20all%20%22{ark_id}%22"
        "&recordSchema=unimarcxchange&maximumRecords=20&startRecord=1"
    )

    resp = rq.get(query_url)
    if resp.status_code != 200:
        return None

    soupe = bs(resp.text, "xml")

    trad_fields = soupe.find_all("mxc:datafield", tag=lambda t: t in ["701", "702"])

    isni_list = []
    for field in trad_fields:
        code4 = field.find("mxc:subfield", {"code": "4"})
        if code4 and code4.text.strip() == "730":
            isni = field.find("mxc:subfield", {"code": "o"})
            if isni:
                isni_list.append(isni.text.strip())

    return isni_list

In [ ]:
df_ark_exploded["ISNI Traducteurs"] = df_ark_exploded["Notices du catalogue"].apply(get_trad_isni)
df_ark_exploded.to_csv("20250926_corpus_traduction_nor-fre_BnF_1900-1950.csv",index= False)

In [ ]:
for col in ["Nom Traducteurs", "ISNI Traducteurs"]:
    df_ark_exploded[col] = df_ark_exploded[col].apply(
        lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else [x])
    )

In [ ]:
def zip_pad(names, isnis):
    names = names or []
    isnis = isnis or []
    maxlen = max(len(names), len(isnis))
    names_p = list(names) + [None] * (maxlen - len(names))
    isnis_p = list(isnis) + [None] * (maxlen - len(isnis))
    return list(zip(names_p, isnis_p))

In [ ]:
df_ark_exploded["pairs"] = df_ark_exploded.apply(
    lambda r: zip_pad(r["Nom Traducteurs"], r["ISNI Traducteurs"]),
    axis=1
)

In [ ]:
df_pairs = df_ark_exploded.explode("pairs").dropna(subset=["pairs"]).copy()

df_pairs[["Nom Traducteurs", "ISNI Traducteurs"]] = pd.DataFrame(
    df_pairs["pairs"].tolist(), index=df_pairs.index
)

df_pairs["Nom Traducteurs"] = df_pairs["Nom Traducteurs"].astype(str).str.strip().replace('None', pd.NA)
def normalize_isni(s):
    if pd.isna(s) or s in [None, "None"]:
        return pd.NA
    s = str(s).strip()

    if s.isdigit():
        return "ISNI" + s

    if s.upper().startswith("ISNI"):
        return s.upper()
    return s

df_pairs["ISNI Traducteurs"] = df_pairs["ISNI Traducteurs"].apply(normalize_isni)


df_pairs = df_pairs.dropna(subset=["Nom Traducteurs"])

In [ ]:
df_traducteurs = (
    df_pairs
    .groupby("Nom Traducteurs", dropna=False)
    .agg(
        ISNI=("ISNI Traducteurs", lambda x: sorted({v for v in x.dropna()})),
        nb_publications=("Notices du catalogue", "count"),
        annees=("Date", lambda x: sorted({v for v in x.dropna()}))
    )
    .reset_index()
)


df_traducteurs.columns = ["Nom Traducteurs", "ISNI", "Nombre de traductions", "Années d'activité"]

df_traducteurs = df_traducteurs.sort_values(
by="Nombre de traductions",
ascending=False
)


df_traducteurs.to_csv("20250926_les_traducteurs_nor-fre_BnF_1900-1950.csv",index= False)